In [9]:
suppressMessages({
    require(Seurat)
    require(dplyr)
    require(igraph)
    require(ggplot2)
    require(ggpubr)
})

In [37]:
# load orthogroups
orthogroups <- read.delim('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/02.gene_relationships/run4/results/Ortho_pipeline/OrthoFinder/Orthogroups/Orthogroups.tsv')
# at least one copy for 4 species
orthogroups <- orthogroups %>% select(c('Orthogroup', 'Pmar', 'Pvit', 'Mmus', 'Hsap'))  %>% 
    filter(Pmar != '' | Pvit != '' | Mmus != '' | Hsap != '')

In [2]:
# get TFs for each species
Hsap_TFs <- read.table('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/07.species_signals/6.TF_vs_species_signals/TFs/Hsap_TFs.txt', header = F)$V1
Mmus_TFs <- read.table('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/07.species_signals/6.TF_vs_species_signals/TFs/Mmus_TFs.name.txt', header = F)$V1
Pvit_TFs <- read.table('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/07.species_signals/6.TF_vs_species_signals/TFs/Pvit.predicted_TFs.txt', header = T)
Pmar_TFs <- read.table('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/07.species_signals/6.TF_vs_species_signals/TFs/Pmar.predicted_TFs.txt', header = T)
Pvit_TFs <- Pvit_TFs[Pvit_TFs$prediction == 'True', 1]
Pmar_TFs <- Pmar_TFs[Pmar_TFs$prediction == 'True', 1]

In [3]:
# get ohnologues and SSD paralogues info
oh_pa_family <- readRDS('Combined.SSD_WGD.pairs.rds')
paralog_gene_type <- readRDS('gene_type.rds')

In [4]:
# gene annotation for gene ID and gene name
Hsap_ID <- read.delim('0.bin/Hsap.info', header = T)
Hsap_ID <- Hsap_ID[Hsap_ID$Gene.type == 'protein_coding', ]
Hsap_ID[Hsap_ID$Gene.name == '', 'Gene.name'] <- Hsap_ID[Hsap_ID$Gene.name == '', 'Gene.stable.ID']

Mmus_ID <- read.delim('0.bin/Mmus.info', header = T)
Mmus_ID <- Mmus_ID[Mmus_ID$Gene.type == 'protein_coding', ]
Mmus_ID[Mmus_ID$Gene.name == '', 'Gene.name'] <- Mmus_ID[Mmus_ID$Gene.name == '', 'Gene.stable.ID']

In [6]:
Hsap <- readRDS('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/02.atlas_final/2.samap/4.final/Hsap.wb.iter_cluster_annotated.rds')
Mmus <- readRDS('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/02.atlas_final/2.samap/4.final/Mmus.wb.iter_cluster_annotated.rds')
Pvit <- readRDS('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/02.atlas_final/2.samap/4.final/Pvit.non_neurons.iter_cluster_annotated.rds')
Pmar <- readRDS('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/02.atlas_final/2.samap/4.final/Pmar.wb.iter_cluster_annotated.rds')

In [11]:
# retain only astrocytes and oligodendrocytes
Idents(Hsap) <- 'Refined family'
Idents(Mmus) <- 'Refined family'
Idents(Pvit) <- 'Refined family'
Idents(Pmar) <- 'Refined family'
Hsap <- subset(Hsap, idents = c('Astrocytes', 'Ependymal cells'))
Mmus <- subset(Mmus, idents = c('Astrocytes', 'Ependymal cells'))
Pvit <- subset(Pvit, idents = c('Astrocytes', 'Ependymal cells'))
Pmar <- subset(Pmar, idents = c('Astrocytes', 'Ependymal cells'))

In [20]:
# huge inbalance between AST and Ependymal cells especially in human and mouse
subset_further <- function(obj, number){
    sampled_df <- obj@meta.data
    sampled_df$cellname <- rownames(sampled_df)
    sampled_df <- sampled_df %>% group_by(`Refined family`) %>% slice_sample(n = number) %>% ungroup()
    obj <- subset(obj, cells = as.character(sampled_df$cellname))
    return(obj)
}

Hsap <- subset_further(Hsap, 5000)
Mmus <- subset_further(Mmus, 5000)
Pvit <- subset_further(Pvit, 5000)
Pmar <- subset_further(Pmar, 5000)

In [25]:
# Find markers between AST and Oligo
get_markers <- function(object, label){
    markers <- FindAllMarkers(object, only.pos = T, verbose = F)
    markers <- markers %>% filter(p_val_adj < 0.05 & avg_log2FC >= 0.58 & pct.1 > 0.1)
    # rank by pct and avg log2FC
    markers <- markers %>% group_by(cluster) %>% 
        arrange(
            desc(pct.1 / (pct.1 + pct.2)),
            desc(avg_log2FC),
            .by_group = TRUE
        ) %>% ungroup() %>% as.data.frame()
    markers$species <- label
    return(markers)
}

In [26]:
# find markers and add gene names or IDs
Hsap_markers <- get_markers(Hsap, 'Hsap')
Mmus_markers <- get_markers(Mmus, 'Mmus')
Pvit_markers <- get_markers(Pvit, 'Pvit')
Pmar_markers <- get_markers(Pmar, 'Pmar')

In [28]:
# add gene name, TFs, ohnologs?, SSD paralogs?
# add gene categories
add_type <- function(markers, label){
    res <- vapply(markers$gene, FUN = function(g){
        tmp <- paralog_gene_type[[label]]
        if (g %in% tmp$gene){
            res <- tmp[tmp$gene == g, 'type']
        } else {
            res <- 'Others'
        }
    }, FUN.VALUE = character(1))
    markers$type <- res
    return(markers)
}

Hsap_markers$gene_name <- Hsap_ID[match(Hsap_markers$gene, Hsap_ID$Gene.stable.ID), 'Gene.name']
Hsap_markers$TF <- Hsap_markers$gene %in% Hsap_TFs
Hsap_markers <- add_type(Hsap_markers, 'Hsap')

Mmus_markers$gene_name <- Mmus_ID[match(Mmus_markers$gene, Mmus_ID$Gene.name), 'Gene.stable.ID']
Mmus_markers$TF <- Mmus_markers$gene %in% Mmus_TFs
Mmus_markers <- add_type(Mmus_markers, 'Mmus')

Pvit_markers$gene_name <- Pvit_markers$gene
Pvit_markers$TF <- Pvit_markers$gene %in% Pvit_TFs
Pvit_markers <- add_type(Pvit_markers, 'Pvit')

Pmar_markers$gene_name <- Pmar_markers$gene
Pmar_markers$TF <- Pmar_markers$gene %in% Pmar_TFs
Pmar_markers <- add_type(Pmar_markers, 'Pmar')

In [29]:
# add family information, get family level ohnologs and SSD paralogues
get_family <- function(genes, label){
    # get family in lists
    g <- graph_from_data_frame(oh_pa_family[[label]][,1:2], directed = FALSE)
    components <- components(g)$membership
    family <- split(names(components), components)
    
    # Find the matching family for the gene
    results <- vapply(genes, FUN = function(gene) {
        res <- lapply(family, function(x) {
            if (gene %in% x) return(x)
            })
        # Filter out NULL values (elements where the gene wasn't found)
        res <- Filter(Negate(is.null), res)
        
        # If a match is found, return the first result; otherwise, return an empty string
        if (length(res) > 0) {
            return(paste0(unlist(res), collapse = ","))
        } else {
            return("")
        }
    }, FUN.VALUE = character(1))
    return(results)
}

In [30]:
Hsap_markers$family <- get_family(Hsap_markers$gene, 'Hsap')
Mmus_markers$family <- get_family(Mmus_markers$gene, 'Mmus')
Pvit_markers$family <- get_family(Pvit_markers$gene, 'Pvit')
Pmar_markers$family <- get_family(Pmar_markers$gene, 'Pmar')

In [38]:
# add orthogroup information
get_orthogroup <- function(markers, species){
    tmp = orthogroups[, c('Orthogroup', species)] %>% filter(species != '') %>% 
        tidyr::separate_rows(species, sep = ", ") %>% as.data.frame()
    tmp <- tmp[match(markers, tmp[[species]]),1]
    return(tmp)
}

Hsap_markers$orthogroup <- get_orthogroup(Hsap_markers$gene, 'Hsap')
Mmus_markers$orthogroup <- get_orthogroup(Mmus_markers$gene, 'Mmus')
Pvit_markers$orthogroup <- get_orthogroup(Pvit_markers$gene, 'Pvit')
Pmar_markers$orthogroup <- get_orthogroup(Pmar_markers$gene, 'Pmar')

Warning message:
“Using an external vector in selections was deprecated in tidyselect 1.1.0.
ℹ Please use `all_of()` or `any_of()` instead.
  # Was:
  data %>% select(species)

  # Now:
  data %>% select(all_of(species))

See <https://tidyselect.r-lib.org/reference/faq-external-vector.html>.”


In [32]:
# function to get ohnologue family with different members used in AST and Epen
get_markers_for_divergence_WGD <- function(markers, label){
    markers <- markers %>% filter(type == 'WGD')
    tmp1 <- unique(unlist(markers %>% filter(cluster == 'Astrocytes') %>% select(family)))
    tmp2 <- unique(unlist(markers %>% filter(cluster == 'Ependymal cells') %>% select(family)))
    tmp1 <- tmp1[tmp1 != '']
    tmp2 <- tmp2[tmp2 != '']
    x <- (table(c(tmp1,tmp2)) == 2)
    interested <- names(x)[x]
    markers <- markers %>% filter(family %in% interested)
    
    cat(paste0(label, ':\nNumber of ohnologue family involved:', length(unique(c(tmp1,tmp2))),
               ';\nNumber of ohnologue family involved only in one of AST and Epen:', sum(table(c(tmp1,tmp2)) == 1),
               ';\nNumber of ohnologue family involved in these two:', sum(table(c(tmp1,tmp2)) == 2), '\n'))
    return(markers)
}

# function to get SSD paralogue family with different members used in AST and Epen
get_markers_for_divergence_SSD <- function(markers, label){
    markers <- markers %>% filter(type == 'SSD')
    tmp1 <- unique(unlist(markers %>% filter(cluster == 'Astrocytes') %>% select(family)))
    tmp2 <- unique(unlist(markers %>% filter(cluster == 'Ependymal cells') %>% select(family)))
    tmp1 <- tmp1[tmp1 != '']
    tmp2 <- tmp2[tmp2 != '']
    x <- (table(c(tmp1,tmp2)) == 2)
    interested <- names(x)[x]
    markers <- markers %>% filter(family %in% interested)
    
    cat(paste0(label, ':\nNumber of SSD paralogue family involved:', length(unique(c(tmp1,tmp2))),
               ';\nNumber of SSD paralogue family involved only in one of AST and Epen:', sum(table(c(tmp1,tmp2)) == 1),
               ';\nNumber of SSD paralogue family involved in these two:', sum(table(c(tmp1,tmp2)) == 2), '\n'))
    return(markers)
}

In [33]:
Hsap_interesting_WGD <- get_markers_for_divergence_WGD(Hsap_markers, 'Hsap')
Hsap_interesting_SSD <- get_markers_for_divergence_SSD(Hsap_markers, 'Hsap')
Mmus_interesting_WGD <- get_markers_for_divergence_WGD(Mmus_markers, 'Mmus')
Mmus_interesting_SSD <- get_markers_for_divergence_SSD(Mmus_markers, 'Mmus')
Pvit_interesting_WGD <- get_markers_for_divergence_WGD(Pvit_markers, 'Pvit')
Pvit_interesting_SSD <- get_markers_for_divergence_SSD(Pvit_markers, 'Pvit')
Pmar_interesting_WGD <- get_markers_for_divergence_WGD(Pmar_markers, 'Pmar')
Pmar_interesting_SSD <- get_markers_for_divergence_SSD(Pmar_markers, 'Pmar')

Hsap:
Number of ohnologue family involved:762;
Number of ohnologue family involved only in one of AST and Epen:693;
Number of ohnologue family involved in these two:69
Hsap:
Number of SSD paralogue family involved:638;
Number of SSD paralogue family involved only in one of AST and Epen:582;
Number of SSD paralogue family involved in these two:56
Mmus:
Number of ohnologue family involved:392;
Number of ohnologue family involved only in one of AST and Epen:367;
Number of ohnologue family involved in these two:25
Mmus:
Number of SSD paralogue family involved:329;
Number of SSD paralogue family involved only in one of AST and Epen:312;
Number of SSD paralogue family involved in these two:17
Pvit:
Number of ohnologue family involved:358;
Number of ohnologue family involved only in one of AST and Epen:347;
Number of ohnologue family involved in these two:11
Pvit:
Number of SSD paralogue family involved:236;
Number of SSD paralogue family involved only in one of AST and Epen:227;
Number of SS

In [93]:
# AST conserved marker family, Oligo conserved marker family
tmp1 = c(
    unique(unlist(Hsap_markers %>% filter(cluster == 'Astrocytes' & TF) %>% select('orthogroup'))),
    unique(unlist(Mmus_markers %>% filter(cluster == 'Astrocytes' & TF) %>% select('orthogroup'))),
    unique(unlist(Pvit_markers %>% filter(cluster == 'Astrocytes' & TF) %>% select('orthogroup'))),
    unique(unlist(Pmar_markers %>% filter(cluster == 'Astrocytes' & TF) %>% select('orthogroup')))
)
tmp2 = c(
    unique(unlist(Hsap_markers %>% filter(cluster == 'Ependymal cells' & TF) %>% select('orthogroup'))),
    unique(unlist(Mmus_markers %>% filter(cluster == 'Ependymal cells' & TF) %>% select('orthogroup'))),
    unique(unlist(Pvit_markers %>% filter(cluster == 'Ependymal cells' & TF) %>% select('orthogroup'))),
    unique(unlist(Pmar_markers %>% filter(cluster == 'Ependymal cells' & TF) %>% select('orthogroup')))
)

tmp1 = names(table(tmp1))[table(tmp1) == 4]
tmp2 = names(table(tmp2))[table(tmp2) == 4]

Vertebrate_AST_TF <- rbind(Hsap_markers, Mmus_markers, Pvit_markers, Pmar_markers) %>% 
        filter(orthogroup %in% tmp1 & TF & cluster == 'Astrocytes')
Vertebrate_Epen_TF <- rbind(Hsap_markers, Mmus_markers, Pvit_markers, Pmar_markers) %>% 
        filter(orthogroup %in% tmp2 & TF & cluster == 'Ependymal cells')

In [94]:
# number of conserved families separting two sister cell-types
length(unique(c(unique(Vertebrate_AST_TF$orthogroup), unique(Vertebrate_Epen_TF$orthogroup))))

[1] 6

In [95]:
write.table(rbind(Vertebrate_AST_TF, Vertebrate_Epen_TF), file = 'Vertebrate_conserved.AST_vs_Epen.TF_orthogroup.txt', 
            quote = F, sep = '\t', row.names= F, col.names = T)

In [97]:
# check TFs and references
#Vertebrate_Epen_TF %>% filter(species == 'Hsap') %>% arrange(orthogroup)
#Vertebrate_Epen_TF %>% filter(species == 'Mmus') %>% arrange(orthogroup)
#Vertebrate_Epen_TF %>% filter(species == 'Pvit') %>% arrange(orthogroup)
#Vertebrate_Epen_TF %>% filter(species == 'Pmar') %>% arrange(orthogroup)
#Vertebrate_AST_TF %>% filter(species == 'Hsap') %>% arrange(orthogroup)
#Vertebrate_AST_TF %>% filter(species == 'Mmus') %>% arrange(orthogroup)
#Vertebrate_AST_TF %>% filter(species == 'Pvit') %>% arrange(orthogroup)
#Vertebrate_AST_TF %>% filter(species == 'Pmar') %>% arrange(orthogroup)

In [14]:
# for figure 2h 
fig2h_AST_Epen <- data.frame(species = rep(c('Hsap', 'Mmus', 'Pvit', 'Pmar'), each = 4),
                             dup = rep(c('WGD', 'WGD', 'SSD', 'SSD'), times = 4),
                             type = rep(c('one', 'both', 'one', 'both'), times = 4),
                             number = c(693, 69, 582, 56, 367, 25, 312, 17, 347, 11, 227, 9, 426, 16, 383, 17))
fig2h_AST_Epen$species <- factor(fig2h_AST_Epen$species, levels = c('Hsap', 'Mmus', 'Pvit', 'Pmar'))
fig2h_AST_Epen$dup <- factor(fig2h_AST_Epen$dup, levels = c('WGD', 'SSD'))

pdf('Fig2h.AST_vs_Epen.case_numbers.pdf', width = 6, height = 6)
fig2h_AST_Epen %>% ggbarplot(x = "type", y = "number", fill = "type", color = "type") + 
        scale_fill_manual(values=c("#C17F9E","#8BACD1"))+
        scale_color_manual(values=c("#C17F9E","#8BACD1"))+
        facet_grid(vars(dup), vars(species))
dev.off()

pdf 
  2